# TP2: Data Preprocessing — Solution

**Institute of Technology of Cambodia (ITC)**

This notebook contains the solution for both problems:

- **Problem 1** — Warm-up with `Data.csv` (imputation, encoding, splitting, scaling)
- **Problem 2** — Titanic dataset (exploration, missing values, redundancy & leakage, outliers, feature engineering, full pipeline)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

---
# Problem 1: Warm-up with `Data.csv`

> The file is **tab-separated**, so it must be loaded with `sep="\t"`.

The cell below recreates `Data.csv` only if it is not already in the working folder, so the notebook runs anywhere. If students have the original file, it is used as-is.

In [2]:
import os

if not os.path.exists("Data.csv"):
    data_txt = """Country\tAge\tSalary\tPurchased
France\t44\t72000\tNo
Spain\t27\t48000\tYes
Germany\t30\t54000\tNo
Spain\t38\t61000\tNo
Germany\t40\t\tYes
France\t35\t58000\tYes
Spain\t\t52000\tNo
France\t48\t79000\tYes
Germany\t50\t83000\tNo
France\t37\t67000\tYes
"""
    with open("Data.csv", "w") as f:
        f.write(data_txt)
    print("Data.csv created.")

Data.csv created.


## Q1. Load and inspect

In [3]:
df = pd.read_csv("Data.csv", sep="\t")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
df.head(10)

Shape: (10, 4)

Data types:
Country          str
Age          float64
Salary       float64
Purchased        str
dtype: object


,Country,Age,Salary,Purchased
0,France,44.0,72000.0,No
1,Spain,27.0,48000.0,Yes
2,Germany,30.0,54000.0,No
3,Spain,38.0,61000.0,No
4,Germany,40.0,NaN,Yes
5,France,35.0,58000.0,Yes
6,Spain,NaN,52000.0,No
7,France,48.0,79000.0,Yes
8,Germany,50.0,83000.0,No
9,France,37.0,67000.0,Yes


In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Numerical  :", num_cols)
print("Categorical:", cat_cols)

**Answer.** The dataset has 10 rows and 4 columns.

- **Numerical:** `Age`, `Salary` (stored as `float64` because they contain `NaN`; a missing value forces a float type).
- **Categorical:** `Country` (nominal, 3 categories) and `Purchased` (binary target: Yes/No).

## Q2. Find missing values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nRows that contain at least one missing value:")
df[df.isnull().any(axis=1)]

In [ ]:
for col in df.columns:
    rows = df.index[df[col].isnull()].tolist()
    if rows:
        print(f"{col:8s}: {len(rows)} missing, at row index {rows}")

**Answer.** Two values are missing: one `Salary` (row index 4, Germany, age 40) and one `Age` (row index 6, Spain, salary 52000). `Country` and `Purchased` are complete.

## Q3. Split features and target

In [ ]:
X = df[["Country", "Age", "Salary"]]
y = df["Purchased"]

print("X shape:", X.shape)
print("y shape:", y.shape)
X

## Q4. Impute missing values (mean vs median)

In [ ]:
X_mean = X.copy()
imputer_mean = SimpleImputer(strategy="mean")
X_mean[["Age", "Salary"]] = imputer_mean.fit_transform(X[["Age", "Salary"]])

print("Values learned by the mean imputer (Age, Salary):", imputer_mean.statistics_)
X_mean

In [ ]:
X_median = X.copy()
imputer_median = SimpleImputer(strategy="median")
X_median[["Age", "Salary"]] = imputer_median.fit_transform(X[["Age", "Salary"]])

comparison = pd.DataFrame({
    "mean":   imputer_mean.statistics_,
    "median": imputer_median.statistics_,
}, index=["Age", "Salary"])
comparison

**Answer.**

| | Mean | Median |
|---|---|---|
| Age (row 6) | 38.78 | 38 |
| Salary (row 4) | 63 777.78 | 61 000 |

Here the two strategies give close values because this small dataset has no extreme values.

**When is the median better?** The mean is pulled toward extreme values; the median is not. The median is the better choice when the distribution is **skewed** or contains **outliers**. Salary is the classic example: one executive earning 1 000 000 would raise the mean salary a lot, and every missing salary would be filled with an unrealistically high value. The median stays at the "typical" employee. The mean is fine for roughly symmetric distributions without outliers.

## Q5. Encode `Country` with one-hot encoding

In [ ]:
ct = ColumnTransformer(
    transformers=[("onehot", OneHotEncoder(sparse_output=False), ["Country"])],
    remainder="passthrough",
)
X_encoded = ct.fit_transform(X_mean)

feature_names = ct.get_feature_names_out()
X_encoded_df = pd.DataFrame(X_encoded, columns=feature_names)
X_encoded_df

**Why France = 0, Spain = 1, Germany = 2 is a bad idea.**
Label encoding gives the categories an **artificial order and distance** that do not exist. A model would read Germany (2) > Spain (1) > France (0), and conclude that "Spain is halfway between France and Germany", or that Germany = 2 × Spain. Linear models, KNN, SVM, and neural networks all use these numbers in arithmetic, so the meaningless order directly changes predictions.

One-hot encoding creates one binary column per country. All countries are at the same distance from each other, and no order is implied. This is the right choice for **nominal** variables.

## Q6. Encode the target

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", le.classes_, "->", le.transform(le.classes_))
print("y encoded:", y_encoded)

`LabelEncoder` sorts the classes alphabetically, so **No → 0** and **Yes → 1**. Label encoding is acceptable for the **target** because the model treats it as class labels, not as a numeric feature.

## Q7. Split the data (80/20, `random_state=1`)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_df, y_encoded, test_size=0.2, random_state=1
)
print("X_train:", X_train.shape, " X_test:", X_test.shape)
X_train

In [ ]:
X_test

**Why split before scaling?** The test set must simulate **new, unseen data**. If the scaler is fitted on the full dataset, its mean and standard deviation already contain information from the test rows. The model is then evaluated on data that indirectly influenced the training, which gives an optimistic and dishonest score. This is called **data leakage**.

The correct order is: **split → fit the scaler on the training set → apply (transform) it to both sets**.

> Note: strictly speaking, the same rule applies to imputation. In Q4 the imputer was fitted on all 10 rows, so the test rows influenced the filled mean. Problem 2 (Q-Pipeline) shows the correct way, where imputation is fitted on the training set only.

## Q8. Scale `Age` and `Salary` with `StandardScaler` (fit on training set only)

In [ ]:
num_features = ["remainder__Age", "remainder__Salary"]

X_train_std = X_train.copy()
X_test_std = X_test.copy()

sc = StandardScaler()
X_train_std[num_features] = sc.fit_transform(X_train[num_features])   # fit + transform on train
X_test_std[num_features]  = sc.transform(X_test[num_features])        # transform only on test

print("Learned mean :", sc.mean_)
print("Learned std  :", sc.scale_)
print("\nTraining set:")
display(X_train_std.round(3))
print("Test set:")
display(X_test_std.round(3))

After standardization, the training columns have mean ≈ 0 and standard deviation = 1. The test values are expressed in the same units (computed with the **training** mean and std), so they are not exactly centered at 0. This is expected.

**Should the one-hot columns also be scaled?** In general, **no**:

- They are already on a small 0/1 scale, comparable to standardized values.
- Scaling them destroys their interpretation ("1 = this country") and gives values like −0.65 / 1.53 that mean nothing to a human.
- Tree-based models do not need scaling at all.

There are exceptions: for distance-based models (KNN, K-Means) or models with regularization, some practitioners scale everything so all features have the same weight. It rarely changes results much. The common practice taught in this course is to scale only the real numerical features.

## Q9. Compare with `MinMaxScaler`

In [ ]:
X_train_mm = X_train.copy()
X_test_mm = X_test.copy()

mm = MinMaxScaler()
X_train_mm[num_features] = mm.fit_transform(X_train[num_features])
X_test_mm[num_features]  = mm.transform(X_test[num_features])

compare = pd.DataFrame({
    "Age_raw":       X_train["remainder__Age"].values,
    "Age_standard":  X_train_std["remainder__Age"].values,
    "Age_minmax":    X_train_mm["remainder__Age"].values,
    "Salary_raw":    X_train["remainder__Salary"].values,
    "Salary_standard": X_train_std["remainder__Salary"].values,
    "Salary_minmax": X_train_mm["remainder__Salary"].values,
}, index=X_train.index).round(3)
print("Training set comparison:")
display(compare)

print("Test set with MinMaxScaler:")
display(X_test_mm.round(3))

**How the values differ.**

| | StandardScaler | MinMaxScaler |
|---|---|---|
| Formula | $z = \dfrac{x - \mu}{\sigma}$ | $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$ |
| Training range | unbounded, centered at 0 (mostly between −2 and 2) | exactly [0, 1] |
| Negative values | yes | no (on training data) |
| Sensitivity to outliers | moderate | high: one extreme value squeezes all the others near 0 |

The **order** of the values is identical; only the scale changes. Test values can fall slightly **outside [0, 1]** with MinMaxScaler if they are smaller or larger than the training min/max.

Rule of thumb: use **StandardScaler** by default (linear/logistic regression, SVM, PCA); use **MinMaxScaler** when a bounded range is required (e.g., neural network inputs, image pixels) and the data has no strong outliers.

---
# Problem 2: Titanic dataset